<a href="https://colab.research.google.com/github/ashycoding/Deep-Learning-Labs/blob/main/lab5_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Easy Seq2Seq Model for Text-to-Text Learning

## Goal of the Lab
- Learn how one sequence is converted into another sequence  
- Example: input text → output text  

## What We Will Learn
- What Seq2Seq (Sequence-to-Sequence) means  
- Difference between Seq2Seq and next-word prediction  
- How to prepare simple input-output text pairs  
- How to build an encoder-decoder model using LSTM  
- How the model maps input text to output text  

## Why This is Easy
- Uses a very small dataset  
- No need to download anything  
- Simple character-level encoding  
- Easy encoder-decoder model  

- Designed for quick understanding in lab sessions


## 1. Import required libraries
We use NumPy for array handling and TensorFlow/Keras for building the encoder-decoder model.


In [1]:
import numpy as np
# Used for numerical operations and handling arrays

import tensorflow as tf
# Main deep learning library used to build and train models

from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
# Input: defines the input layer of the model
# LSTM: processes sequential data and learns patterns
# Dense: fully connected layer for output prediction
# Embedding: converts text indices into dense vector representations

from tensorflow.keras.models import Model
# Used to build custom models like encoder-decoder (Seq2Seq)

print("TensorFlow version:", tf.__version__)
# Displays installed TensorFlow version

TensorFlow version: 2.19.0


# Step 2: Create a Tiny Dataset

## Idea
- The model learns to map one text sequence to another  
- input sequence → output sequence  

## Example (English → French)
- hello → bonjour  
- thank you → merci  
- good night → bonne nuit  

- This is only for understanding the Seq2Seq workflow, not for building a real translator

In [2]:
input_texts = [
    "hello","thank you","good night","yes","no","good morning","good evening","how are you","see you","welcome",
    "sorry","please","excuse me","i am fine","what is your name","nice to meet you","good afternoon","take care","well done","good luck"
]

target_texts = [
    "bonjour","merci","bonne nuit","oui","non","bonjour","bonsoir","comment ca va","a bientot","bienvenue",
    "desole","s'il vous plait","excusez moi","je vais bien","comment tu t'appelles","ravi de vous rencontrer","bon apres midi","prends soin de toi","bien joue","bonne chance"
]

print("Number of pairs:", len(input_texts))
print("Input texts:", input_texts)
print("Target texts:", target_texts)

Number of pairs: 20
Input texts: ['hello', 'thank you', 'good night', 'yes', 'no', 'good morning', 'good evening', 'how are you', 'see you', 'welcome', 'sorry', 'please', 'excuse me', 'i am fine', 'what is your name', 'nice to meet you', 'good afternoon', 'take care', 'well done', 'good luck']
Target texts: ['bonjour', 'merci', 'bonne nuit', 'oui', 'non', 'bonjour', 'bonsoir', 'comment ca va', 'a bientot', 'bienvenue', 'desole', "s'il vous plait", 'excusez moi', 'je vais bien', "comment tu t'appelles", 'ravi de vous rencontrer', 'bon apres midi', 'prends soin de toi', 'bien joue', 'bonne chance']


# Step 3: Add Start and End Tokens

- The decoder needs signals to start and stop generating text  

## Tokens Used
- `\t` → start token (beginning of output)  
- `\n` → end token (end of output)  

- These help the model understand when to begin and finish prediction

In [3]:

target_texts = ["\t" + text + "\n" for text in target_texts]

print("Target texts with start/end symbols:")
for t in target_texts:
    print(repr(t))


Target texts with start/end symbols:
'\tbonjour\n'
'\tmerci\n'
'\tbonne nuit\n'
'\toui\n'
'\tnon\n'
'\tbonjour\n'
'\tbonsoir\n'
'\tcomment ca va\n'
'\ta bientot\n'
'\tbienvenue\n'
'\tdesole\n'
"\ts'il vous plait\n"
'\texcusez moi\n'
'\tje vais bien\n'
"\tcomment tu t'appelles\n"
'\travi de vous rencontrer\n'
'\tbon apres midi\n'
'\tprends soin de toi\n'
'\tbien joue\n'
'\tbonne chance\n'


# Step 4: Build Character Vocabulary

- We use character-level encoding instead of words  
- Each character is assigned a unique number  

- This keeps the model simple and easy to train  
- Suitable for small datasets in lab experiments

In [4]:

input_chars = sorted(list(set("".join(input_texts))))
target_chars = sorted(list(set("".join(target_texts))))

input_char_to_index = {char: i + 1 for i, char in enumerate(input_chars)}
target_char_to_index = {char: i + 1 for i, char in enumerate(target_chars)}

input_index_to_char = {i: char for char, i in input_char_to_index.items()}
target_index_to_char = {i: char for char, i in target_char_to_index.items()}

num_encoder_tokens = len(input_chars) + 1
num_decoder_tokens = len(target_chars) + 1

print("Input characters :", input_chars)
print("Target characters:", target_chars)
print("Number of encoder tokens:", num_encoder_tokens)
print("Number of decoder tokens:", num_decoder_tokens)


Input characters : [' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y']
Target characters: ['\t', '\n', ' ', "'", 'a', 'b', 'c', 'd', 'e', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'x', 'z']
Number of encoder tokens: 24
Number of decoder tokens: 25



## 5. Find maximum sequence lengths

Since all sequences in a batch must have the same length, we identify:
- maximum encoder input length
- maximum decoder input length


In [5]:

max_encoder_seq_length = max(len(text) for text in input_texts)
max_decoder_seq_length = max(len(text) for text in target_texts)

print("Max encoder sequence length:", max_encoder_seq_length)
print("Max decoder sequence length:", max_decoder_seq_length)


Max encoder sequence length: 17
Max decoder sequence length: 25


# Step 6: Prepare Model Inputs and Outputs

## Encoder Input
- Contains the input sequence converted into indices  

## Decoder Input
- Begins with the start token (`\t`)  
- Followed by the target sequence characters  

## Decoder Output
- Contains the target sequence shifted by one step  
- The model learns to predict the next character  

- This shifting helps the model learn correct sequence generation

In [6]:

encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length), dtype="int32")
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length), dtype="int32")
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    # Encoder input
    for t, char in enumerate(input_text):
        encoder_input_data[i, t] = input_char_to_index[char]

    # Decoder input and decoder target
    for t, char in enumerate(target_text):
        decoder_input_data[i, t] = target_char_to_index[char]
        if t > 0:
            decoder_target_data[i, t - 1, target_char_to_index[char]] = 1.0

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)


Encoder input shape: (20, 17)
Decoder input shape: (20, 25)
Decoder target shape: (20, 25, 25)


# Step 7: Build the Encoder

- The encoder processes the input sequence step by step  

## Outputs
- hidden state (`state_h`)  
- cell state (`state_c`)  

- These states store learned information  
- They are passed to the decoder to generate the output sequence

In [7]:

latent_dim = 64

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_dim=num_encoder_tokens, output_dim=16, mask_zero=True, name="encoder_embedding")(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]


# Step 8: Build the Decoder

- The decoder generates the output sequence step by step  

## How it Works
- Takes the previous output character as input  
- Uses encoder states (`state_h`, `state_c`) as starting context  
- Predicts the next character at each step  

- This process continues until the end token is generated

In [8]:

decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding_layer = Embedding(input_dim=num_decoder_tokens, output_dim=16, mask_zero=True, name="decoder_embedding")
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)



## 9. Define the full Seq2Seq model

The full training model takes:
- encoder input sequence
- decoder input sequence

and learns to predict:
- decoder output sequence


In [9]:

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 16)  │        384 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 16)  │        400 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     20,736 │ encoder_embeddin… │
│                     │ (None, 64),       │            │ not_equal[0][0]   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     20,736 │ decoder_embeddin… │
│                     │ 64), (None, 64),  │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 25)  │      1,625 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 43,881 (171.41 KB)

 Trainable params: 43,881 (171.41 KB)

 Non-trainable params: 0 (0.00 B)


## 10. Train the model

Because the dataset is very small, we train for more epochs so that the model can learn the tiny mappings well.


In [10]:

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=100,
    verbose=1
)

# Print final accuracy
print("Final Training Accuracy:", history.history['accuracy'][-1])

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.0984 - loss: 2.9352
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1299 - loss: 2.8878
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1299 - loss: 2.7866
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1260 - loss: 2.7006
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1457 - loss: 2.6296
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1142 - loss: 2.5643
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1575 - loss: 2.5090
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1457 - loss: 2.4910
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1575 - loss: 2.4559
Epoch 10/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1890 - loss: 2.4510
Epoch 11/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2008 - loss: 2.4516
Epoch 12/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


## 11. Build inference models for prediction

During training, the full target sequence is known.

During testing, we generate one character at a time:
1. encode the input
2. start decoder with the start token
3. predict next character
4. feed predicted character back to decoder
5. stop when end token is produced


In [11]:

# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inf = decoder_embedding_layer(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf, initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)



## 12. Define decoding function

This function converts a new input text into an output text using the trained encoder and decoder.


In [12]:

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1), dtype="int32")
    target_seq[0, 0] = target_char_to_index["\t"]

    decoded_sentence = ""

    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_index_to_char.get(sampled_token_index, "")

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            break

        decoded_sentence += sampled_char

        target_seq = np.zeros((1, 1), dtype="int32")
        target_seq[0, 0] = sampled_token_index

        states_value = [h, c]

    return decoded_sentence



## 13. Test the Seq2Seq model

Now we check whether the model has learned the simple sequence mapping.


In [13]:

for seq_index in range(len(input_texts)):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print(f"Input: {input_texts[seq_index]}  -->  Predicted Output: {decoded_sentence}")


Input: hello  -->  Predicted Output: bonjour
Input: thank you  -->  Predicted Output: merci
Input: good night  -->  Predicted Output: bonne nui
Input: yes  -->  Predicted Output: oui
Input: no  -->  Predicted Output: non
Input: good morning  -->  Predicted Output: bonjour
Input: good evening  -->  Predicted Output: bonjour
Input: how are you  -->  Predicted Output: comment ca va
Input: see you  -->  Predicted Output: a bientot
Input: welcome  -->  Predicted Output: bienvenue
Input: sorry  -->  Predicted Output: desole
Input: please  -->  Predicted Output: soui
Input: excuse me  -->  Predicted Output: excusez moi
Input: i am fine  -->  Predicted Output: je vais bien
Input: what is your name  -->  Predicted Output: comment t ta tappelle
Input: nice to meet you  -->  Predicted Output: comment t ta palle
Input: good afternoon  -->  Predicted Output: bon ar rsien
Input: take care  -->  Predicted Output: penceus mie
Input: well done  -->  Predicted Output: bien joue
Input: good luck  -->  Pr


## 14. Try your own input

This helper function allows students to test a word from the known training vocabulary.


In [14]:
def encode_input_text(text):
    seq = np.zeros((1, max_encoder_seq_length), dtype="int32")
    for t, char in enumerate(text[:max_encoder_seq_length]):
        if char in input_char_to_index:
            seq[0, t] = input_char_to_index[char]
    return seq

# Examples (updated as per new dataset)
test_words = ["hello", "thank you", "good night", "yes", "goodbye", "good morning"]

for word in test_words:
    encoded = encode_input_text(word)
    print(f"Input: {word}  -->  Output: {decode_sequence(encoded)}")

Input: hello  -->  Output: bonjour
Input: thank you  -->  Output: merci
Input: good night  -->  Output: bonne nui
Input: yes  -->  Output: oui
Input: goodbye  -->  Output: bonnenue
Input: good morning  -->  Output: bonjour
